In [1]:
# -----------------------------------------
# 
# move era5 files to zarr store
# 
# -----------------------------------------
import xarray as xr
import fsspec
import re
import math
import os
from collections import defaultdict
import zarr

# Configuration
input_glob = 'carbonplan-carbon-removal/era5/preprocessed_data/*.nc'
output_root = "s3://carbonplan-carbon-removal/era5/preprocessed_zarr/"
batch_size = 20

# Set up S3 filesystem
fs = fsspec.filesystem('s3')

# Step 1: Discover all .nc files
all_paths = fs.glob(input_glob)
all_paths = ['s3://' + path for path in all_paths]

# Step 2: Group by filename prefix
var_groups = defaultdict(list)
pattern = re.compile(r".*/([a-zA-Z0-9_]+)-\d{4}-\d{2}\.nc$")  # matches varname-YYYY-MM.nc

for path in all_paths:
    match = pattern.match(path)
    if match:
        varname = match.group(1)
        var_groups[varname].append(path)


In [2]:
# --- remove vars we've already saved on aws... 
del var_groups['2m_temperature'], var_groups['evaporation_from_vegetation_transpiration']
del var_groups['potential_evaporation'], var_groups['runoff'], var_groups['skin_reservoir_content']
del var_groups['total_evaporation'], var_groups['total_precipitation']
del var_groups['volumetric_soil_water_layer_1'], var_groups['volumetric_soil_water_layer_2']

for key, value in var_groups.items():
    print(f"Key: {key}")

Key: geopotential
Key: land_sea_mask
Key: soil_type
Key: volumetric_soil_water_layer_3
Key: volumetric_soil_water_layer_4


In [3]:
# Step 3: Process each variable group
for label_varname, paths in var_groups.items():
    if label_varname in ['geopotential', 'land_sea_mask', 'soil_type']: 
        continue
    
    paths = sorted(paths)
    print(f"\n🚀 Processing variable group: {label_varname} ({len(paths)} files)")

    # 🔍 Step 3a: Get actual variable name from first file
    sample_ds = xr.open_dataset(fs.open(paths[0]), engine='h5netcdf')
    candidate_vars = list(sample_ds.data_vars)

    if len(candidate_vars) != 1:
        print(f"  ⚠️ Multiple or no variables in {paths[0]}: {candidate_vars}. Skipping.")
        continue

    real_varname = candidate_vars[0]
    sample_ds.close()
    print(f"  ✅ Identified variable '{real_varname}' from sample file")

    # Prepare output path and batch size
    n_batches = math.ceil(len(paths) / batch_size)
    zarr_path = os.path.join(output_root, f'{label_varname}.zarr')

    for i in range(n_batches):
        batch = paths[i * batch_size: (i + 1) * batch_size]
        print(f"  ⏳ Batch {i+1}/{n_batches} with {len(batch)} files...")

        # Open all datasets in the batch
        open_files = [xr.open_dataset(fs.open(f), engine='h5netcdf', chunks={}) for f in batch]
        # Combine the datasets
        ds = xr.combine_by_coords(open_files, combine_attrs="drop_conflicts")

        # 🔐 Keep only the real variable
        if real_varname in ds:
            ds = ds[[real_varname]]
        else:
            print(f"  ⚠️ Variable '{real_varname}' missing in batch. Skipping batch.")
            continue

        # Rechunk
        chunk_dims = list(ds.dims.keys())
        chunk_dict = {dim: min(100, ds.dims[dim]) for dim in chunk_dims}
        ds = ds.chunk(chunk_dict)

        # Write or append to Zarr
        mode = 'w' if i == 0 else 'a'
        append_dim = chunk_dims[0]  # assume first dim (likely 'time' or 'valid_time')
        if mode == 'w':
            ds.to_zarr(zarr_path, mode=mode, consolidated=False)
        if mode == 'a': 
            ds.to_zarr(zarr_path, mode=mode, append_dim=append_dim, consolidated=False)
        ds.close()

    # Final metadata consolidation
    print(f"  ✅ Consolidating metadata for {label_varname}")
    store = fsspec.get_mapper(zarr_path)
    zarr.consolidate_metadata(store)



🚀 Processing variable group: volumetric_soil_water_layer_3 (120 files)
  ✅ Identified variable 'swvl3' from sample file
  ⏳ Batch 1/6 with 20 files...


/tmp/ipykernel_371/1091432786.py:42: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  chunk_dims = list(ds.dims.keys())
/tmp/ipykernel_371/1091432786.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  chunk_dict = {dim: min(100, ds.dims[dim]) for dim in chunk_dims}


  ⏳ Batch 2/6 with 20 files...


/tmp/ipykernel_371/1091432786.py:42: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  chunk_dims = list(ds.dims.keys())
/tmp/ipykernel_371/1091432786.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  chunk_dict = {dim: min(100, ds.dims[dim]) for dim in chunk_dims}


  ⏳ Batch 3/6 with 20 files...


...

  ⏳ Batch 5/6 with 20 files...


/tmp/ipykernel_371/1091432786.py:42: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  chunk_dims = list(ds.dims.keys())
/tmp/ipykernel_371/1091432786.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  chunk_dict = {dim: min(100, ds.dims[dim]) for dim in chunk_dims}


  ⏳ Batch 6/6 with 20 files...


/tmp/ipykernel_371/1091432786.py:42: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  chunk_dims = list(ds.dims.keys())
/tmp/ipykernel_371/1091432786.py:43: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  chunk_dict = {dim: min(100, ds.dims[dim]) for dim in chunk_dims}


  ✅ Consolidating metadata for volumetric_soil_water_layer_4


In [ ]:
# ------------------------------------------